# Regime Change Detection Feature Engineering

**Author:** Codex

**Description:** This notebook loads merged BTC spot and perpetual market data, then builds return-, volatility-, and ATR-based features for downstream regime change analysis.


In [8]:
# Import pandas for tabular data work and numpy for numerical feature calculations.
import pandas as pd
import numpy as np

In [12]:
# Load the merged hourly BTC dataset and parse the timestamp column as datetimes.
df = pd.read_csv("merged_btc_data.csv", parse_dates=['timestamp'])
# Use timestamp as the index so rolling features align naturally over time.
df.set_index('timestamp', inplace=True)
# Inspect a few random rows to confirm the dataset loaded as expected.
df.sample(3)

,open_spot,high_spot,low_spot,close_spot,volume_spot,cost_spot,open_perp,high_perp,low_perp,close_perp,volume_perp,cost_perp,index_price,interest_8h,interest_1h,prev_index_price
timestamp,,,,,,,,,,,,,,,,
2024-08-08 15:00:00+00:00,58834.00,59468.00,58719.00,59468.00,10.7701,635969.416600,58966.5,59609.5,58674.5,59483.5,1043.114633,61754090.0,58961.87,0.000005,3.865911e-08,57280.96
2025-01-19 02:00:00+00:00,104525.00,105405.00,104525.00,104830.00,1.4942,157005.591000,104439.0,105363.0,104414.5,104814.5,319.130943,33489700.0,104381.30,0.000235,2.999130e-05,104030.08
2024-02-20 15:00:00+00:00,51921.85,51952.67,51383.17,51614.56,1.9049,98373.888537,52077.0,52114.5,51403.0,51693.5,2328.313101,120242010.0,52045.72,0.000253,3.119955e-05,52844.88


In [13]:
# Compute log returns plus absolute and squared features for the core price series.
df = df.assign(
    # Spot close-to-close log return.
    return_close_spot=lambda x: np.log(x["close_spot"] / x["close_spot"].shift(1)),
    # Perpetual close-to-close log return.
    return_close_perp=lambda x: np.log(x["close_perp"] / x["close_perp"].shift(1)),
    # Index price log return.
    return_index_price=lambda x: np.log(x["index_price"] / x["index_price"].shift(1)),
    # Absolute spot return as a simple volatility proxy.
    abs_return_close_spot=lambda x: x["return_close_spot"].abs(),
    # Absolute perpetual return as a simple volatility proxy.
    abs_return_close_perp=lambda x: x["return_close_perp"].abs(),
    # Absolute index return as a simple volatility proxy.
    abs_return_index_price=lambda x: x["return_index_price"].abs(),
    # Squared spot return for variance-style features.
    sq_return_close_spot=lambda x: x["return_close_spot"] ** 2,
    # Squared perpetual return for variance-style features.
    sq_return_close_perp=lambda x: x["return_close_perp"] ** 2,
    # Squared index return for variance-style features.
    sq_return_index_price=lambda x: x["return_index_price"] ** 2,
)

In [ ]:
# Compute rolling standard deviations for all return features over 24h and 72h windows.
# These are the return-derived series for which we want short- and medium-horizon volatility features.
return_cols = [
    "return_close_spot",
    "return_close_perp",
    "return_index_price",
    "abs_return_close_spot",
    "abs_return_close_perp",
    "abs_return_index_price",
    "sq_return_close_spot",
    "sq_return_close_perp",
    "sq_return_index_price",
]

# Use 24 hourly observations for short-term volatility.
window_24h = 24
# Use 72 hourly observations for medium-term volatility.
window_72h = 72

for col in return_cols:
    # Rolling standard deviation over the last 24 hours.
    df[f"std_24h_{col}"] = df[col].rolling(window_24h).std()
    # Rolling standard deviation over the last 72 hours.
    df[f"std_72h_{col}"] = df[col].rolling(window_72h).std()

df.filter(regex="^(std_24h_|std_72h_)").head()


In [ ]:
# Compute ATR features for spot and perpetual markets.
# ATR starts from the True Range (TR), which captures the largest one-period move.
# For spot, TR is the maximum of:
# 1. current high minus current low,
# 2. absolute distance from current high to previous close,
# 3. absolute distance from current low to previous close.
tr_spot = pd.concat(
    [
        df["high_spot"] - df["low_spot"],
        (df["high_spot"] - df["close_spot"].shift(1)).abs(),
        (df["low_spot"] - df["close_spot"].shift(1)).abs(),
    ],
    axis=1,
).max(axis=1)

# Apply the same True Range definition to perpetual futures prices.
tr_perp = pd.concat(
    [
        df["high_perp"] - df["low_perp"],
        (df["high_perp"] - df["close_perp"].shift(1)).abs(),
        (df["low_perp"] - df["close_perp"].shift(1)).abs(),
    ],
    axis=1,
).max(axis=1)

# ATR is the rolling mean of True Range over the last 24 hourly observations for spot.
df["ATR_24h_spot"] = tr_spot.rolling(24).mean()
# Longer-horizon spot ATR using a 72-hour rolling window.
df["ATR_72h_spot"] = tr_spot.rolling(72).mean()
# Normalize ATR by spot price so the feature is scale-free across price regimes.
df["ATR_24h_spot_norm"] = df["ATR_24h_spot"] / df["close_spot"]
# Same normalization for the 72-hour spot ATR.
df["ATR_72h_spot_norm"] = df["ATR_72h_spot"] / df["close_spot"]
# 24-hour ATR for perpetuals from the rolling mean of perpetual True Range.
df["ATR_24h_perp"] = tr_perp.rolling(24).mean()
# 72-hour ATR for perpetuals.
df["ATR_72h_perp"] = tr_perp.rolling(72).mean()
# Normalize perpetual ATR by perpetual close to express volatility in relative terms.
df["ATR_24h_perp_norm"] = df["ATR_24h_perp"] / df["close_perp"]
# Same normalization for the 72-hour perpetual ATR.
df["ATR_72h_perp_norm"] = df["ATR_72h_perp"] / df["close_perp"]

df[["ATR_24h_spot", "ATR_72h_spot", "ATR_24h_spot_norm", "ATR_72h_spot_norm", "ATR_24h_perp", "ATR_72h_perp", "ATR_24h_perp_norm", "ATR_72h_perp_norm"]].head()


In [ ]:
# Placeholder cell for additional exploratory analysis or visualizations.


In [ ]:
# Placeholder cell for model preparation, labeling, or later experiments.
